In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format('json')\
    .option("multiline", "true")\
    .option("inferSchema", "true")\
    .option("header", "true")\
    .load("/Volumes/labuser14944459_1777732916/stream/data/")
display(df)

In [0]:
df.schema

In [0]:
schema = StructType(
    [
        StructField(
            "customer",
            StructType(
                [
                    StructField(
                        "address",
                        StructType(
                            [
                                StructField("city", StringType(), True),
                                StructField("country", StringType(), True),
                                StructField("postal_code", StringType(), True),
                            ]
                        ),
                        True,
                    ),
                    StructField("customer_id", LongType(), True),
                    StructField("email", StringType(), True),
                    StructField("name", StringType(), True),
                ]
            ),
            True,
        ),
        StructField(
            "items",
            ArrayType(
                StructType(
                    [
                        StructField("item_id", StringType(), True),
                        StructField("price", DoubleType(), True),
                        StructField("product_name", StringType(), True),
                        StructField("quantity", LongType(), True),
                    ]
                ),
                True,
            ),
            True,
        ),
        StructField(
            "metadata",
            ArrayType(
                StructType(
                    [
                        StructField("key", StringType(), True),
                        StructField("value", StringType(), True),
                    ]
                ),
                True,
            ),
            True,
        ),
        StructField("order_id", StringType(), True),
        StructField(
            "payment",
            StructType(
                [
                    StructField("method", StringType(), True),
                    StructField("transaction_id", StringType(), True),
                ]
            ),
            True,
        ),
        StructField("timestamp", StringType(), True),
    ]
)

In [0]:
df.printSchema()

In [0]:
display(df)

In [0]:
df = spark.read.format('json')\
    .option("multiline", "true")\
    .schema(schema)\
    .option("header", "true")\
    .load("/Volumes/labuser14944459_1777732916/stream/data/")
display(df)

In [0]:
df.withColumn('name', col("customer").getField("email"))\
    .withColumn("customer_id", col("customer").getField("customer_id"))\
    .withColumn("email", col("customer.email"))\
    .withColumn("address", col("customer.address"))\
    .display()

In [0]:
df = df.select(
    "order_id",
    "timestamp",
    col("customer.customer_id"),
    col("customer.name"),
    col("customer.email"),
    col("customer.address.city"),
    col("customer.address.country"),
    col("customer.address.postal_code"),
    col("payment.method"),
    col("payment.transaction_id"),
    explode_outer("items").alias("items"),
    explode_outer("metadata").alias("metadata")
)

df = df.select(
    "*",
    col("items.item_id"),
    col("items.price"),
    col("items.product_name"),
    col("items.quantity"),
    col("metadata.key"),
    col("metadata.value")
).drop("items", "metadata")

In [0]:
display(df)

## Read Streaming Data

In [0]:
# spark.conf.set("spark.sql.streaming.schema.inference.maxFiles",")

In [0]:
df = spark.readStream.format('json')\
    .option("multiline", "true")\
    .schema(schema)\
    .option("header", "true")\
    .load("/Volumes/labuser14944459_1777732916/stream/data/")

In [0]:
df = df.select(
    "order_id",
    "timestamp",
    col("customer.customer_id"),
    col("customer.name"),
    col("customer.email"),
    col("customer.address.city"),
    col("customer.address.country"),
    col("customer.address.postal_code"),
    col("payment.method"),
    col("payment.transaction_id"),
    explode_outer("items").alias("items"),
    explode_outer("metadata").alias("metadata")
)

df = df.select(
    "*",
    col("items.item_id"),
    col("items.price"),
    col("items.product_name"),
    col("items.quantity"),
    col("metadata.key"),
    col("metadata.value")
).drop("items", "metadata")

In [0]:
display(df)

In [0]:
%sql
-- DROP TABLE labuser14944459_1777732916.stream.customers;

In [0]:
df.writeStream.format("delta")\
    .outputMode('append')\
    .option("checkpointLocation", "/Volumes/labuser14944459_1777732916/stream/files/checkpoint")\
    .toTable("labuser14944459_1777732916.stream.customers")

In [0]:
df_stream = spark.table("labuser14944459_1777732916.stream.customers")
display(df)